# Ejercicio 3: Modelo Vectorial y TF-IDF

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`.
- Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`.

### Paso 1: Calcular la matriz TF-IDF para el corpus `01_corpus_turismo_500.txt`

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Carga del corpus — cada línea es un documento
with open('../data/01_corpus_turismo_500.txt', 'r', encoding='utf-8') as f:
    corpus_turismo = [line.strip() for line in f if line.strip()]

print(f"Documentos: {len(corpus_turismo)}")

# Construir matriz TF-IDF
vectorizer_turismo = TfidfVectorizer()
tfidf_turismo = vectorizer_turismo.fit_transform(corpus_turismo)
vocab_turismo = vectorizer_turismo.get_feature_names_out()

# Visualizar como DataFrame (subconjunto)
df_tfidf_turismo = pd.DataFrame(tfidf_turismo.toarray(), columns=vocab_turismo)
print(df_tfidf_turismo.head())

Documentos: 500
   2000  agua  amazonía  arquitectura  artesanía     atrae  atraen  auténtico  \
0   0.0   0.0       0.0           0.0   0.341664  0.000000     0.0        0.0   
1   0.0   0.0       0.0           0.0   0.000000  0.000000     0.0        0.0   
2   0.0   0.0       0.0           0.0   0.000000  0.352958     0.0        0.0   
3   0.0   0.0       0.0           0.0   0.000000  0.000000     0.0        0.0   
4   0.0   0.0       0.0           0.0   0.000000  0.000000     0.0        0.0   

   aventura  aves  ...  típica        un       una  vida  vilcabamba  visitan  \
0       0.0   0.0  ...     0.0  0.000000  0.000000   0.0    0.000000      0.0   
1       0.0   0.0  ...     0.0  0.000000  0.000000   0.0    0.000000      0.0   
2       0.0   0.0  ...     0.0  0.000000  0.240057   0.0    0.352958      0.0   
3       0.0   0.0  ...     0.0  0.175574  0.000000   0.0    0.000000      0.0   
4       0.0   0.0  ...     0.0  0.229707  0.000000   0.0    0.000000      0.0   

   visitan

### Paso 2: Construir el corpus `Gutenberg 1000`

El corpus `Gutenberg 1000` es un corpus compuesto por 1000 libros de Gutenberg Project

In [15]:
import os

path = '../data/gutenberg/data/'
files = os.listdir(path)[:1000]

corpus_gutenberg = []
doc_names = []

for filename in files:
    filepath = os.path.join(path, filename)
    try:
        with open(filepath, encoding='utf-8', errors='ignore') as f:
            corpus_gutenberg.append(f.read())
            doc_names.append(filename)
    except Exception as e:
        print(f"Error {filename}: {e}")

print(f"Corpus Gutenberg 1000:")
print(f"  Documentos cargados : {len(corpus_gutenberg)}")
print(f"  Fragmento doc[0]    : {corpus_gutenberg[0][:100]}")


Corpus Gutenberg 1000:
  Documentos cargados : 1000
  Fragmento doc[0]    : Produced by Chuck Greif and the Online Distributed
Proofreading Team at http://www.pgdp.net (This fi


### Paso 3: Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# min_df=2: descarta términos que aparecen en menos de 2 documentos (ruido/errores tipográficos)
vectorizer_gutenberg = TfidfVectorizer(
    min_df=2,
    max_df=0.5,  # descarta términos que aparecen en más del 50% de los documentos (poco informativos)
)

# Calcular la matriz TF-IDF
tfidf_gutenberg = vectorizer_gutenberg.fit_transform(corpus_gutenberg)
vocab_gutenberg = vectorizer_gutenberg.get_feature_names_out()

# Verificación
print(f"Documentos procesados: {tfidf_gutenberg.shape[0]}")
print(f"Vocabulario filtrado: {len(vocab_gutenberg)}")
print(f"Ejemplo vocabulario: {vocab_gutenberg[:10]}")


Documentos procesados: 1000
Vocabulario filtrado: 364358
Ejemplo vocabulario: ['00' '000' '0008' '000_' '001' '002' '003' '004' '005' '006']


### Paso 4: Programar una función `buscar()` para el corpus `Gutenberg 1000`

In [19]:
from sklearn.metrics.pairwise import cosine_similarity

def buscar(query: str, top_k: int = 10) -> pd.DataFrame:
  query_vec = vectorizer_gutenberg.transform([query])  # (1, vocab)
  sim_scores = cosine_similarity(query_vec, tfidf_gutenberg).flatten()  # (num_docs,)
  top_indices = sim_scores.argsort()[-top_k:][::-1]  # índices de los top_k documentos

  for idx in top_indices:
    if sim_scores[idx] > 0:
      print(f"Documento: {doc_names[idx]} - Similitud: {sim_scores[idx]:.4f}")
      print(f"Fragmento: {corpus_gutenberg[idx][:200]}...\n")
      
# Ejemplo de búsqueda
buscar("adventure sea",top_k=5)

Documento: sir_francis_drake_his_voyage_1595.txt - Similitud: 0.0067
Fragmento: THIS “O-P BOOK” IS AN AUTHORIZED REPRINT OF THE ORIGINAL EDITION,
PRODUCED BY MICROFILM-XEROX BY UNIVERSITY MICROFILMS, INC., ANN ARBOR,
MICHIGAN, 1961




                             WORKS ISSUED BY...

Documento: la_moza_de_cantaro.txt - Similitud: 0.0043
Fragmento: Produced by Juliet Sutherland, Chuck Greif and the Online
Distributed Proofreading Team at http://www.pgdp.net









LA MOZA DE CÁNTARO

POR

LOPE DE VEGA

_EDITED WITH INTRODUCTION AND NOTES_

BY
...

Documento: el_estudiante_de_salamanca_and_other_selections.txt - Similitud: 0.0028
Fragmento: Produced by Stan Goodman, Miranda van de Heijning, Renald
Levesque and the Online Distributed Proofreading Team.







[Illustration: D. JOSÉ DE ESPRONCEDA]

  ESPRONCEDA

  EL ESTUDIANTE
  DE SALAMA...

Documento: dona_clarines_y_manana_de_sol.txt - Similitud: 0.0022
Fragmento: E-text prepared by Stan Goodman, Nieves Rodríguez, and the Project
Gut